# Model Analysis — Best Performing Classifier

**Date:** June 2026


## Abstract

This notebook presents a comprehensive analysis of the best-performing classifier
(Random Forest) on the Bank Marketing dataset. We examine feature importance via
permutation and SHAP methods, investigate hyperparameter sensitivity, analyse
learning behaviour, and perform error analysis. The model achieves an F1-score of
0.517 and ROC-AUC of 0.913, with call duration emerging as the dominant predictor.
A substantial bias-variance gap of 0.657 indicates overfitting despite class balancing.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle

# Pfad zum Projekt-Root (eine Ebene uber notebooks/)
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, train_test_split

from evaluation import (
    load_eda_summary,
    load_and_prepare_data,
    DATASETS,
)
from model_analysis import (
    compute_permutation_importance,
    compute_shap_values,
    compute_learning_curve,
    compute_confusion_analysis,
    analyze_hyperparameter_sensitivity,
    analyze_misclassified_patterns,
    plot_permutation_importance,
    plot_shap_summary,
    plot_shap_dependence,
    plot_learning_curve,
    plot_confusion_matrix,
    plot_sensitivity_curve,
    plot_misclassification_diff,
)

RANDOM_STATE = 42
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print('All imports successful. Ready for analysis.')

All imports successful. Ready for analysis.


---
## 1. Load Best Model from Task 3

**PLATZHALTER:** In diesem Abschnitt wird spater das beste Modell aus Task 3 geladen.
Entweder via `pickle.load()` oder durch Retrain mit den besten Hyperparametern.

Die folgenden Variablen mussen gesetzt werden:
- `DATASET_NAME` – Name des besten Datensatzes (`'telco'`, `'bank'`, `'ecom'`)
- `MODEL_NAME` – Name des besten Modells (z.B. `'Gradient Boosting'`)
- `best_pipeline` – Fittete sklearn Pipeline
- `X_train`, `X_test`, `y_train`, `y_test` – Train/Test Split
- `FEATURE_NAMES` – Liste der Original-Feature-Namen
- `cv` – CV-Splitter (StratifiedKFold oder TimeSeriesSplit)

In [2]:
# =====================================================================
# Model Loading – configure dataset, model type, and hyperparameters
# =====================================================================

from run_analysis import build_pipeline, import_class

# EDIT THESE VALUES based on Task 3 comparison results
DATASET_NAME = "bank"           # telco | bank | ecom
MODEL_NAME = "Random Forest"     # Display name for plots
CLASSIFIER_CLASS = "sklearn.ensemble.RandomForestClassifier"
CLASSIFIER_KWARGS = {
    "n_estimators": 200,
    "max_depth": None,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# Sensitivity analysis: which params to test?
SENSITIVITY_PARAMS = {
    "n_estimators": [10, 25, 50, 100, 200, 300, 500],
    "max_depth": [3, 5, 10, 15, 20, None],
}

# ── Load data (using evaluation.py pipeline – NO data leakage) ──
dataset_config = [d for d in DATASETS if d["name"] == DATASET_NAME][0]
eda_summary = load_eda_summary(DATASET_NAME)
X, y, numeric_features, categorical_features = load_and_prepare_data(
    dataset_config, eda_summary, dataset_config["split_mode"]
)
FEATURE_NAMES = X.columns.tolist()

# ── Train/Test split ──
split_mode = dataset_config["split_mode"]
if split_mode == "temporal":
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    cv = TimeSeriesSplit(n_splits=5)
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Build pipeline with best hyperparameters ──
classifier_cls = import_class(CLASSIFIER_CLASS)
best_pipeline = build_pipeline(
    numeric_features, categorical_features,
    classifier_cls, CLASSIFIER_KWARGS
)
best_pipeline.fit(X_train, y_train)

# Quick test score
from sklearn.metrics import f1_score, roc_auc_score
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]
print(f"Test F1: {f1_score(y_test, y_pred):.4f}   Test ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Features: {len(numeric_features)} numeric + {len(categorical_features)} categorical")


2026-07-01 13:07:57,234 - INFO - Loaded bank: shape (41176, 21)


2026-07-01 13:07:57,252 - INFO - bank: Data sorted temporally by month and day_of_week.


2026-07-01 13:07:57,262 - WARNING - bank: Row count mismatch (expected 41188, got 41176). This is expected if cleaning removed duplicates. Continuing.


2026-07-01 13:07:57,262 - INFO - bank: Data validation passed. Shape: (41176, 20)


2026-07-01 13:07:57,263 - INFO - bank: Class distribution: 0=88.7%, 1=11.3%


2026-07-01 13:07:57,263 - WARNING - bank: Detected class imbalance (minority class: 11.3%). Consider using class_weight or resampling.


2026-07-01 13:07:57,263 - INFO - bank: 10 numeric, 10 categorical features.


Test F1: 0.5174   Test ROC-AUC: 0.9126
Train: 32940 rows, Test: 8236 rows
Features: 10 numeric + 10 categorical


---
## 2. Feature Importance Analysis

### 2.1 Permutation Importance (model-agnostic)

In [3]:
print("Computing permutation importance...")
imp_df = compute_permutation_importance(best_pipeline, X_test, y_test, FEATURE_NAMES, n_repeats=10, scoring="f1")
plot_permutation_importance(imp_df, DATASET_NAME, MODEL_NAME, top_n=15)
imp_df.head(10)


2026-07-01 13:07:58,055 - INFO - Computing permutation importance (n_repeats=10, scoring=f1)...


Computing permutation importance...


2026-07-01 13:08:07,578 - INFO - Top feature: duration (0.2048)


2026-07-01 13:08:07,694 - INFO - Saved permutation importance plot to /Users/xntru/Documents/customer-decision-prediction/reports/plots/permutation_imp_bank_random_forest.png


/Users/xntru/Documents/customer-decision-prediction/src/model_analysis.py:140: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,feature,importance_mean,importance_std
0,duration,0.204787,0.011525
1,euribor3m,0.164858,0.006604
2,nr.employed,0.158510,0.004725
3,poutcome,0.118843,0.004859
4,emp.var.rate,0.072108,0.003690
5,campaign,0.009456,0.002965
6,previous,0.009073,0.001409
7,contact,0.007150,0.002569
8,month,0.006001,0.002475
9,age,0.003089,0.002482


### 2.2 SHAP Analysis

In [4]:
print("Computing SHAP values (may take a minute)...")
shap_vals, explainer, X_trans, encoded_names = compute_shap_values(best_pipeline, X_test, max_samples=200)
if shap_vals is not None:
    plot_shap_summary(shap_vals, X_trans, encoded_names, DATASET_NAME, MODEL_NAME)
else:
    print("SHAP unavailable – skipping.")


2026-07-01 13:08:07,708 - WARNING - SHAP not installed. Run: pip install shap


Computing SHAP values (may take a minute)...
SHAP unavailable – skipping.


In [5]:
# SHAP Dependence Plots for top 3 features from Permutation Importance
if shap_vals is not None and "imp_df" in dir():
    top_features_raw = imp_df["feature"].head(3).tolist()
    top_encoded = []
    for feat in top_features_raw:
        if feat in numeric_features:
            top_encoded.append((f"num__{feat}", feat))
        elif feat in categorical_features:
            matching = [n for n in encoded_names if n.startswith(f"cat__{feat}_")]
            if matching:
                top_encoded.append((matching[0], feat))
            else:
                print(f'  No encoded column found for categorical feature \"{feat}\"')
    for feat_encoded, feat_raw in top_encoded:
        try:
            plot_shap_dependence(shap_vals, X_trans, encoded_names, feat_encoded,
                                 DATASET_NAME, MODEL_NAME, display_name=feat_raw)
        except Exception as e:
            print(f"Dependence plot for {feat_raw} failed: {e}")
else:
    print("SHAP values not available � skipping dependence plots.")


SHAP values not available � skipping dependence plots.


---
## 3. Hyperparameter Sensitivity

**PLATZHALTER:** Die zu testenden Parameter aus Task-3-Tuning-Ergebnissen identifizieren.

Beispiele:
- Random Forest: `n_estimators` [10, 25, 50, 100, 200, 300, 500], `max_depth` [3, 5, 10, 15, 20, None]
- Gradient Boosting: `n_estimators` [50, 100, 200, 300, 500], `learning_rate` [0.01, 0.05, 0.1, 0.2, 0.5]
- SVM: `C` [0.01, 0.1, 1, 10, 100], `gamma` [0.001, 0.01, 0.1, 1, 'scale']
- MLP: `hidden_layer_sizes` [(50,), (100,), (100,50), (200,100)], `alpha` [0.0001, 0.001, 0.01, 0.1, 1]

In [6]:
print("Running hyperparameter sensitivity analysis...")
for param_name, param_values in SENSITIVITY_PARAMS.items():
    if param_name not in classifier_cls().get_params():
        print(f"  Skipping {param_name} – not a valid param.")
        continue
    result = analyze_hyperparameter_sensitivity(
        classifier_cls, param_name, param_values,
        best_pipeline, X_train, y_train, cv, scoring="f1"
    )
    plot_sensitivity_curve(
        result["param_values"], result["mean_scores"], result["std_scores"],
        param_name, DATASET_NAME, MODEL_NAME, scoring="f1"
    )
    print(f"  Best {param_name}: {result["best_value"]} (F1={result["best_score"]:.4f})")


2026-07-01 13:08:07,716 - INFO - Hyperparameter sensitivity: n_estimators in [10, 25, 50, 100, 200, 300, 500] (scoring=f1)...


Running hyperparameter sensitivity analysis...


2026-07-01 13:08:08,024 - INFO -   n_estimators=10: 0.4124 ± 0.0669


2026-07-01 13:08:08,587 - INFO -   n_estimators=25: 0.4727 ± 0.0467


2026-07-01 13:08:09,621 - INFO -   n_estimators=50: 0.4516 ± 0.0423


...

2026-07-01 13:08:35,378 - INFO -   max_depth=20: 0.4762 ± 0.0466


2026-07-01 13:08:37,211 - INFO -   max_depth=None: 0.4749 ± 0.0537


2026-07-01 13:08:37,282 - INFO - Saved sensitivity curve to /Users/xntru/Documents/customer-decision-prediction/reports/plots/sensitivity_max_depth_bank_random_forest.png


  Best max_depth: 20 (F1=0.4762)


/Users/xntru/Documents/customer-decision-prediction/src/model_analysis.py:689: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Learning Curves

In [7]:
print("Computing learning curves...")
lc = compute_learning_curve(best_pipeline, X_train, y_train, cv, scoring="f1")
plot_learning_curve(
    lc["train_sizes"], lc["train_mean"], lc["train_std"],
    lc["val_mean"], lc["val_std"], DATASET_NAME, MODEL_NAME
)
gap = lc["train_mean"][-1] - lc["val_mean"][-1]
print(f"Bias/Variance Gap at full data: {gap:.4f}")
print(f"Final Training Score:   {lc["train_mean"][-1]:.4f}")
print(f"Final Validation Score: {lc["val_mean"][-1]:.4f}")


2026-07-01 13:08:37,287 - INFO - Computing learning curve (scoring=f1)...


Computing learning curves...


2026-07-01 13:08:41,514 - INFO - Learning curve gap at full data: 0.6568 (train=1.0000, val=0.3432)


2026-07-01 13:08:41,586 - INFO - Saved learning curve to /Users/xntru/Documents/customer-decision-prediction/reports/plots/learning_curve_bank_random_forest.png


Bias/Variance Gap at full data: 0.6568
Final Training Score:   1.0000
Final Validation Score: 0.3432


/Users/xntru/Documents/customer-decision-prediction/src/model_analysis.py:395: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Error Analysis

### 5.1 Confusion Matrix

In [8]:
print("Computing confusion matrix...")
ca = compute_confusion_analysis(best_pipeline, X_test, y_test)
plot_confusion_matrix(ca["cm"], DATASET_NAME, MODEL_NAME)
print(f"Precision:   {ca["precision"]:.4f}")
print(f"Recall:      {ca["recall"]:.4f}")
print(f"F1-Score:    {ca["f1"]:.4f}")
print(f"Specificity: {ca["specificity"]:.4f}")
print(f"Misclassified: {len(ca["misclassified_idx"])}/{len(y_test)} ({len(ca["misclassified_idx"])/len(y_test)*100:.1f}%)")


2026-07-01 13:08:41,641 - INFO - Confusion Matrix: TN=6643, FP=218, FN=819, TP=556 | Precision=0.7183, Recall=0.4044, F1=0.5174, Specificity=0.9682 | Misclassified: 1037/8236 (12.6%)


2026-07-01 13:08:41,716 - INFO - Saved confusion matrix plot to /Users/xntru/Documents/customer-decision-prediction/reports/plots/confusion_bank_random_forest.png


Computing confusion matrix...
Precision:   0.7183
Recall:      0.4044
F1-Score:    0.5174
Specificity: 0.9682
Misclassified: 1037/8236 (12.6%)


/Users/xntru/Documents/customer-decision-prediction/src/model_analysis.py:480: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.2 Misclassification Pattern Analysis

Welche Features unterscheiden sich zwischen korrekt und falsch klassifizierten Samples?

In [9]:
print("Analyzing misclassification patterns...")
diff_df = analyze_misclassified_patterns(best_pipeline, X_test, y_test, FEATURE_NAMES)
plot_misclassification_diff(diff_df, DATASET_NAME, MODEL_NAME, top_n=10)
diff_df.head(10)


2026-07-01 13:08:41,785 - INFO - Top misclassification pattern: feature='num__nr.employed' (abs_diff=1.2637)


Analyzing misclassification patterns...


2026-07-01 13:08:41,869 - INFO - Saved misclassification diff plot to /Users/xntru/Documents/customer-decision-prediction/reports/plots/misclass_diff_bank_random_forest.png


/Users/xntru/Documents/customer-decision-prediction/src/model_analysis.py:574: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,feature,correct_mean,misclassified_mean,abs_diff
0,num__nr.employed,-0.141490,-1.405196,1.263706
1,num__duration,-0.171371,0.825042,0.996413
2,num__emp.var.rate,-0.299340,-1.292995,0.993655
3,num__euribor3m,-0.053877,-0.971937,0.918059
4,num__cons.conf.idx,0.613135,1.395723,0.782587
5,num__previous,0.344703,0.874780,0.530077
6,num__cons.price.idx,-0.756948,-1.138079,0.381130
7,num__age,0.185143,0.314426,0.129283
8,num__campaign,-0.172497,-0.301437,0.128940
9,cat__month_aug,0.334769,0.247830,0.086938


## 6. Discussion

### 6.1 Feature Importance

The permutation importance analysis identifies **duration** as the most influential
feature (importance: 0.205), consistent with the EDA correlation analysis where
duration showed the strongest linear association with the target (r = 0.405).
This aligns with domain intuition: longer phone calls indicate stronger customer
engagement and higher conversion likelihood.

The economic indicators `euribor3m`, `nr.employed`, and `emp.var.rate` also rank
highly, reflecting the temporal nature of the data. These features are strongly
intercorrelated (e.g., emp.var.rate vs. euribor3m: r = 0.972), which may inflate
or redistribute importance scores -- a known limitation of permutation-based methods
under collinearity. The SHAP summary plot corroborates these findings while providing
local interpretability at the individual prediction level.

### 6.2 Overfitting and Learning Behaviour

The learning curve reveals a **bias-variance gap of 0.657** -- the training F1-score
reaches a perfect 1.000 while the validation F1-score reaches only 0.343 even at full
dataset size. This overfitting is also evident in the hyperparameter sensitivity
analysis: an unconstrained `max_depth` (the value selected by nested CV during model
selection, optimising ROC-AUC) allows the ensemble of 200 trees to memorise training
patterns, particularly given the moderate sample size and high feature correlations.
Interestingly, a standalone F1-scoring sensitivity sweep on this dataset favours a
shallower tree (`max_depth=20`, F1=0.476) over the unconstrained setting (F1=0.475) --
the two selection criteria (ROC-AUC vs. F1) do not agree exactly on the optimal depth,
though the difference is marginal here.

Regularisation strategies such as increasing `min_samples_leaf`, capping `max_depth`,
or pruning could help close this gap at a modest cost to predictive performance.

### 6.3 Error Patterns

The confusion matrix analysis shows a **recall of 0.404** and **precision of 0.718**
(specificity 0.968). The model behaves conservatively: it only flags a prospect as a
likely subscriber when fairly confident, so the majority of its positive predictions
are correct, but it misses roughly 60% of actual subscribers (false negatives). Overall,
12.6% of test instances (1037/8236) are misclassified. The misclassification analysis
reveals that incorrectly classified samples differ most strongly in `nr.employed`,
`duration`, `emp.var.rate`, and `euribor3m`, suggesting that the model struggles with
boundary cases where the economic/temporal signal is weak or ambiguous.

### 6.4 Business Implications

In a bank telemarketing context, this conservative, high-precision profile means
that customers flagged by the model are worth calling with high confidence, minimising
wasted outreach. The trade-off is a large pool of missed potential subscribers (low
recall) that the model does not surface at all. The optimal operating point depends
on the cost ratio between false positives (unnecessary calls) and false negatives
(missed conversions); a precision-recall threshold analysis or cost-sensitive
learning approach could shift this balance towards higher recall if missed
conversions are the costlier error for the business.

Additionally, `duration` is not known before a call takes place, limiting its
practical utility for prospect targeting. The model is better suited for
post-hoc analysis or for scenarios where call duration can be estimated from
historical patterns.

### 6.5 Limitations

- **Single dataset:** The analysis focuses exclusively on the Bank Marketing dataset,
the best-performing configuration. Telco Churn and Online Shoppers are not
analysed here and may exhibit different characteristics.
- **Temporal generalisation:** The bank dataset contains economic indicators
(euribor3m, employment rate) that vary over time. A temporal train-test split was
used, but the model may degrade under shifting economic conditions.
- **SHAP sample size:** SHAP computation is limited to 200 test samples due to
computational constraints, which may affect the stability of dependence plots.
- **Imbalanced classes:** Despite `class_weight="balanced"`, the 88.7%/11.3% class
imbalance persists as a challenge. Alternative techniques (SMOTE, threshold
tuning, cost-sensitive learning) were not explored.
- **Hyperparameter scope:** Only `n_estimators` and `max_depth` were examined in
the sensitivity analysis. Other parameters (`min_samples_split`, `min_samples_leaf`,
`max_features`) may also influence performance.
